# TR-CodeBench — Analyse des résultats

Ce notebook analyse les résultats d'un ou plusieurs runs OpenRouter de TR-CodeBench.  
Il est conçu pour être **lisible sans connaître le code** : chaque section commence par une explication des métriques et comment lire le graphique.

**Flux de lecture recommandé :**
1. Vue d'ensemble (section 2) — combien de modèles ? combien réussissent ?
2. Classement des modèles (section 3) — qui fait quoi ?
3. Difficulté des items (section 4) — quelles tâches résistent ?
4. Anatomie du score (section 5) — d'où vient la note finale ?
5. Productive Divergence (section 6) — les modèles innovent-ils vraiment ?
6. True Regime (section 7) — les solutions sont-elles asymptotiquement correctes ?
7. Analyse des échecs (section 8) — pourquoi certains modèles plantent ?

## 0. Setup WSL / Conda

Ce notebook est prévu pour être lancé depuis VS Code connecté à WSL avec un kernel Conda local au projet.

Une fois par machine WSL, créer l'environnement depuis un terminal WSL à la racine du dépôt :

```bash
conda create -y -p ./.conda python=3.11
conda run -p ./.conda python -m pip install -U ipykernel numpy pandas matplotlib seaborn
conda run -p ./.conda python -m ipykernel install --user --name tr-codebench --display-name "TR-CodeBench (.conda WSL)"
```

Ensuite sélectionner le kernel **TR-CodeBench (.conda WSL)** dans VS Code/Jupyter.  
Si l'environnement existe déjà mais que Jupyter affiche une erreur `ipykernel`, relancer les deux commandes `pip install` et `ipykernel install` ci-dessus.


In [ ]:
from __future__ import annotations
import hashlib
import os, sys, re
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from matplotlib.gridspec import GridSpec

# Style global
sns.set_theme(style='whitegrid', font_scale=1.05)

# Libelles connus. Les modeles non listes recoivent un libelle lisible automatiquement.
MODEL_LABELS = {
    'poolside/laguna-m.1:free': 'Laguna M.1',
    'openai/gpt-oss-20b': 'GPT-OSS 20B',
    'openai/gpt-4o': 'GPT-4o',
    'openai/gpt-4o-mini': 'GPT-4o mini',
    'openai/gpt-4.1': 'GPT-4.1',
    'openai/gpt-4.1-mini': 'GPT-4.1 mini',
    'openai/o3': 'o3',
    'openai/o3-mini': 'o3 mini',
    'openai/o4-mini': 'o4 mini',
    'anthropic/claude-3.5-sonnet': 'Claude 3.5 Sonnet',
    'anthropic/claude-3.7-sonnet': 'Claude 3.7 Sonnet',
    'anthropic/claude-sonnet-4': 'Claude Sonnet 4',
    'anthropic/claude-opus-4': 'Claude Opus 4',
    'google/gemini-2.0-flash-001': 'Gemini 2.0 Flash',
    'google/gemini-2.5-flash': 'Gemini 2.5 Flash',
    'google/gemini-2.5-pro': 'Gemini 2.5 Pro',
    'deepseek/deepseek-chat': 'DeepSeek Chat',
    'deepseek/deepseek-chat-v3-0324': 'DeepSeek V3',
    'deepseek/deepseek-r1': 'DeepSeek R1',
    'meta-llama/llama-3.1-405b-instruct': 'Llama 3.1 405B',
    'meta-llama/llama-3.3-70b-instruct': 'Llama 3.3 70B',
    'meta-llama/llama-4-maverick': 'Llama 4 Maverick',
    'meta-llama/llama-4-scout': 'Llama 4 Scout',
    'mistralai/mistral-large': 'Mistral Large',
    'mistralai/mistral-small-24b-instruct-2501': 'Mistral Small 24B',
    'mistralai/codestral-2501': 'Codestral',
    'qwen/qwen3-235b-a22b-2507': 'Qwen3 235B',
    'qwen/qwen3-32b': 'Qwen3 32B',
    'qwen/qwen-2.5-coder-32b-instruct': 'Qwen2.5 Coder 32B',
    'x-ai/grok-3': 'Grok 3',
    'x-ai/grok-3-mini': 'Grok 3 mini',
    'nvidia/nemotron-3-nano-30b-a3b:free': 'Nemotron Nano 30B',
    'cohere/command-r-plus': 'Command R+',
    'moonshotai/kimi-k2': 'Kimi K2',
    'z-ai/glm-4.5': 'GLM-4.5',
}

# Couleurs preferees pour les modeles historiques, puis palette dynamique pour le reste.
PREFERRED_COLORS = {
    'poolside/laguna-m.1:free': '#4C72B0',
    'openai/gpt-oss-20b': '#DD8452',
    'qwen/qwen3-235b-a22b-2507': '#55A868',
    'mistralai/mistral-small-24b-instruct-2501': '#C44E52',
    'nvidia/nemotron-3-nano-30b-a3b:free': '#8172B3',
}
COLOR_POOL = (
    sns.color_palette('tab20', 20).as_hex()
    + sns.color_palette('tab20b', 20).as_hex()
    + sns.color_palette('tab20c', 20).as_hex()
    + sns.color_palette('husl', 32).as_hex()
)
PALETTE: dict[str, str] = {}


def _stable_color(model: str, used: set[str] | None = None) -> str:
    used = used or set()
    start = int(hashlib.sha256(model.encode('utf-8')).hexdigest(), 16) % len(COLOR_POOL)
    for offset in range(len(COLOR_POOL)):
        candidate = COLOR_POOL[(start + offset) % len(COLOR_POOL)]
        if candidate not in used:
            return candidate
    return COLOR_POOL[start]


def build_model_palette(models) -> dict[str, str]:
    """Construit une palette stable pour n'importe quel nombre de modeles."""
    palette: dict[str, str] = {}
    used: set[str] = set()
    for model in dict.fromkeys(models):
        if model in PREFERRED_COLORS:
            palette[model] = PREFERRED_COLORS[model]
            used.add(palette[model])
    for model in dict.fromkeys(models):
        if model not in palette:
            palette[model] = _stable_color(model, used)
            used.add(palette[model])
    return palette


def refresh_model_style(models) -> dict[str, str]:
    global PALETTE
    PALETTE = build_model_palette([m for m in models if pd.notna(m)])
    return PALETTE


def _auto_label(model: str) -> str:
    name = str(model).split('/')[-1].replace(':free', '')
    name = re.sub(r'-(?:20\d{2}|\d{4})(?:-\d{2}){0,2}$', '', name)
    name = re.sub(r'-(?:instruct|preview|latest|free)$', '', name)
    return name.replace('-', ' ').replace('_', ' ').title()[:28]


def short(model):
    return MODEL_LABELS.get(model, _auto_label(model))


def color(model):
    if model not in PALETTE:
        PALETTE[model] = PREFERRED_COLORS.get(model, _stable_color(model, set(PALETTE.values())))
    return PALETTE[model]

# Projet root
def find_root(start: Path) -> Path:
    for c in [start, *start.parents]:
        if (c/'pyproject.toml').exists() and (c/'datasets'/'curated').exists():
            return c
    raise RuntimeError('TR-CodeBench root not found')

ROOT = find_root(Path.cwd().resolve())
for p in (str(ROOT), str(ROOT/'src')):
    if p not in sys.path:
        sys.path.insert(0, p)

print(f'Root : {ROOT}')
print(f'Python : {sys.executable}')
print(f'Kernel attendu : {ROOT / ".conda"}')


: 

## 1. Chargement des données

Le notebook charge automatiquement **tous les fichiers JSONL** dans `reports/openrouter_runs/`.  
Pour analyser un run spécifique, modifier la variable `RESULTS_GLOB` ci-dessous.

**Colonnes clés :**

| Colonne | Type | Signification |
|---|---|---|
| `score` | 0–1 | Score final agrégé |
| `correctness_score` | 0 ou 1 | 1 = passe tous les tests publics ET cachés |
| `optimization_score` | 0 ou 1 | 1 = correct ET dans le bon régime de complexité |
| `pd_score` | 0–1 | Score de Productive Divergence (paradigme différent de l'oracle) |
| `paradigm_distance` | 0–1 | Distance structurelle entre le paradigme candidat et l'oracle |
| `productivity_score` | 0–1 | Efficacité × robustesse |
| `originality_score` | 0–1 | 1 − similarité tokenielle avec l'oracle (anti-plagiat) |
| `salieri_overlap` | 0–1 | Jaccard tokeniel candidat/oracle (1 = copie exacte) |
| `genuine_divergence` | bool | True = paradigme algorithmique vraiment différent |
| `complexity_ratio_ok` | bool | True = ratio temps large/petit dans la borne attendue |
| `hallucination_flag` | bool | True = violation statique, crash, ou hidden tests échoués |

In [ ]:
# ── Paramètre : changer ici pour un run précis ────────────────────────────────
#RESULTS_GLOB = 'reports/openrouter_runs/openrouter_eval_*.jsonl'   # tous les runs
RESULTS_GLOB = 'reports/openrouter_runs/openrouter_eval_20260520T184534Z.jsonl'  # un seul run

paths = sorted((ROOT/'.').glob(RESULTS_GLOB))
if not paths:
    raise FileNotFoundError(f'Aucun fichier JSONL trouvé : {RESULTS_GLOB}')

frames = []
for p in paths:
    df_p = pd.read_json(p, lines=True)
    rid = re.search(r'openrouter_eval_(\d{8}T\d{6}Z)', p.name)
    df_p['run_id'] = rid.group(1) if rid else p.stem
    frames.append(df_p)

df = pd.concat(frames, ignore_index=True)

# Normalisation numérique
num_cols = ['score','correctness_score','optimization_score','pd_score',
            'paradigm_distance','productivity_score','originality_score',
            'public_pass_rate','hidden_pass_rate','pbt_pass_rate',
            'salieri_overlap','complexity_ratio','latency_seconds',
            'prompt_tokens','completion_tokens','total_tokens']
for c in num_cols:
    if c in df.columns: df[c] = pd.to_numeric(df[c], errors='coerce')

bool_cols = ['genuine_divergence','complexity_ratio_ok','hallucination_flag',
             'static_violation','crash','timeout']
for c in bool_cols:
    if c in df.columns: df[c] = df[c].fillna(False).astype(bool)
    else: df[c] = False

if 'originality_score' not in df.columns and 'salieri_overlap' in df.columns:
    df['originality_score'] = 1.0 - df['salieri_overlap']
if 'pd_score' not in df.columns:
    df['pd_score'] = df.get('pd_candidate', pd.Series(False)).astype(float) * 0.15

df['model_short'] = df['model'].map(short)
df['is_error']    = df['api_error'].notna() if 'api_error' in df.columns else False
df_ok = df[~df['is_error']].copy()

model_order = (
    df_ok.groupby('model')['score'].mean()
    .sort_values(ascending=False).index.tolist()
)
refresh_model_style(model_order or sorted(df['model'].dropna().unique()))

print(f'Runs chargés   : {df["run_id"].nunique()}')
print(f'Lignes totales : {len(df)}  |  Réponses valides : {len(df_ok)}')
print(f'Modèles : {sorted(df["model"].unique())}')
print(f'Items   : {sorted(df["item_id"].unique())}')

## 2. Vue d'ensemble — KPIs globaux

**Que montre cette section ?**  
Un résumé en chiffres du run : combien de solutions ont été soumises, combien sont correctes, quel est le taux de réussite global et combien font preuve de Productive Divergence.

**Comment lire les métriques :**
- **Taux de correction** : part des solutions qui passent 100% des tests publics ET des tests cachés
- **Taux PD** : part des solutions correctes dont le paradigme algorithmique diffère de l'oracle
- **Hallucinations** : solutions qui crashent, violent des imports, ou échouent sur les tests cachés malgré les tests publics passés

In [ ]:
total      = len(df_ok)
n_correct  = int((df_ok['correctness_score'] == 1.0).sum())
n_pd       = int((df_ok['pd_score'] > 0).sum())
n_halluc   = int(df_ok['hallucination_flag'].sum())
n_errors   = int(df['is_error'].sum())
mean_score = df_ok['score'].mean()

fig, axes = plt.subplots(1, 5, figsize=(16, 3))
fig.suptitle(f"Run {df['run_id'].iloc[0]}  —  Vue d'ensemble", fontsize=13, y=1.02)

cards = [
    ('Soumissions\nvalides',  total,                     f'{n_errors} erreurs API',              '#4C72B0'),
    ('Solutions\ncorrectes',  f'{n_correct}/{total}',    f'{n_correct/total*100:.0f}% taux',      '#55A868'),
    ('Score moyen',           f'{mean_score:.2f}',       'sur 1.0',                               '#DD8452'),
    ('Productive\nDivergence',f'{n_pd}',                 f'sur {n_correct} correctes',            '#8172B3'),
    ('Hallucinations',        n_halluc,                  f'{n_halluc/total*100:.0f}% taux',       '#C44E52'),
]

for ax, (label, value, sub, col) in zip(axes, cards):
    ax.set_facecolor(col + '18')
    ax.text(0.5, 0.65, str(value), ha='center', va='center',
            fontsize=26, fontweight='bold', color=col, transform=ax.transAxes)
    ax.text(0.5, 0.30, label, ha='center', va='center',
            fontsize=11, color='#333333', transform=ax.transAxes)
    ax.text(0.5, 0.10, sub, ha='center', va='center',
            fontsize=9, color='#888888', transform=ax.transAxes)
    for spine in ax.spines.values(): spine.set_visible(False)
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.show()

## 3. Classement des modèles

**Que montrent ces graphiques ?**  
Les barres groupées comparent chaque modèle sur 4 dimensions. Le dot plot (droite) montre la relation entre correction et PD : un modèle idéal serait en haut à droite (correct ET divergent).

**Comment lire les barres :**
- `Correction` (vert) — passe-t-il tous les tests ?
- `Optimisation` (orange) — est-il dans le bon régime de complexité ?
- `PD Score` (violet) — explore-t-il un paradigme différent de l'oracle ?
- `Score final` (bleu) — note globale (formule : 0.50×correct + 0.20×pbt + 0.15×optim + 0.15×pd − 0.25×halluc)

In [ ]:
model_stats = (
    df_ok.groupby('model')
    .agg(
        score        =('score', 'mean'),
        correctness  =('correctness_score', 'mean'),
        optimization =('optimization_score', 'mean'),
        pd_score     =('pd_score', 'mean'),
        hallucination=('hallucination_flag', 'mean'),
        n            =('item_id', 'count'),
    )
    .sort_values('score', ascending=True)
    .reset_index()
)
model_stats['label'] = model_stats['model'].map(short)

fig, (ax_bar, ax_dot) = plt.subplots(1, 2, figsize=(15, 4.5))

# Barres groupées
metrics   = ['correctness', 'optimization', 'pd_score', 'score']
colors_m  = ['#55A868', '#DD8452', '#8172B3', '#4C72B0']
labels_m  = ['Correction', 'Optimisation', 'PD Score', 'Score final']
x = np.arange(len(model_stats))
w = 0.2
for i, (m, col, lab) in enumerate(zip(metrics, colors_m, labels_m)):
    bars = ax_bar.barh(x + i*w, model_stats[m], w, label=lab, color=col, alpha=0.85)
    for bar, val in zip(bars, model_stats[m]):
        if val > 0.04:
            ax_bar.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                        f'{val:.2f}', va='center', fontsize=8, color='#333')

ax_bar.set_yticks(x + w*1.5)
ax_bar.set_yticklabels(model_stats['label'])
ax_bar.set_xlim(0, 1.15)
ax_bar.set_xlabel('Score moyen')
ax_bar.set_title('Scores par modèle')
ax_bar.legend(loc='lower right', fontsize=9)
ax_bar.axvline(1.0, color='#aaa', linestyle='--', lw=0.8)

# Dot plot correction vs PD
for _, row in model_stats.iterrows():
    col = color(row['model'])
    ax_dot.scatter(row['correctness'], row['pd_score'], s=120, color=col,
                   zorder=5, edgecolors='white', linewidth=1.5)
    ax_dot.annotate(row['label'], (row['correctness'], row['pd_score']),
                    xytext=(6, 4), textcoords='offset points', fontsize=9, color=col)

ax_dot.set_xlabel('Taux de correction')
ax_dot.set_ylabel('PD Score moyen')
ax_dot.set_title('Correction vs Productive Divergence')
ax_dot.set_xlim(-0.05, 1.10)
ax_dot.set_ylim(-0.02, max(0.30, model_stats['pd_score'].max()*1.3))
ax_dot.axhline(0, color='#ddd', lw=0.8)
ax_dot.text(0.02, ax_dot.get_ylim()[1]*0.95, 'Haut = diverge du paradigme oracle',
            fontsize=8, color='#888', style='italic')

plt.tight_layout()
plt.show()

print('\n── Tableau de synthèse par modèle ──')
display(
    model_stats[['label','n','correctness','optimization','pd_score','score','hallucination']]
    .rename(columns={'label':'Modèle','n':'N','correctness':'Correction',
                     'optimization':'Optimisation','pd_score':'PD Score',
                     'score':'Score final','hallucination':'Halluc.'})
    .round(3).reset_index(drop=True)
)

## 4. Difficulté des items (tâches)

**Que montrent ces graphiques ?**  
La heatmap montre le score de chaque modèle sur chaque tâche. Le barplot montre la difficulté intrinsèque de chaque item (score moyen tous modèles).

**Comment lire la heatmap :**
- **Vert foncé** ≥ 0.85 : solution correcte et dans le bon régime de complexité
- **Jaune** ≈ 0.60–0.84 : solution partiellement correcte ou dans le mauvais régime
- **Rouge** = 0 : mauvaise solution, crash, ou erreur API
- **— (N/A)** : pas de réponse API (ex. rate limit 429)

**Interprétation :** Une colonne rouge = modèle défaillant (erreur système ou incapable). Une ligne uniformément jaune = tâche difficile pour tous les modèles.

In [ ]:
item_order = sorted(df_ok['item_id'].unique())

pivot = df_ok.pivot_table(index='item_id', columns='model', values='score', aggfunc='mean')
pivot = pivot.reindex(index=item_order, columns=model_order)
pivot.columns = [short(m) for m in pivot.columns]

item_means = df_ok.groupby('item_id')['score'].mean().reindex(item_order)

fig, (ax_heat, ax_item) = plt.subplots(1, 2, figsize=(15, 5),
                                        gridspec_kw={'width_ratios': [3, 1]})

# Heatmap
mask = pivot.isna()
sns.heatmap(
    pivot.fillna(-0.05), ax=ax_heat,
    cmap='RdYlGn', vmin=0, vmax=1,
    linewidths=0.5, linecolor='white',
    annot=pivot.applymap(lambda v: f'{v:.2f}' if pd.notna(v) else '—'),
    fmt='', annot_kws={'size': 9},
    mask=mask, cbar_kws={'label': 'Score'}
)
for row_i, item in enumerate(pivot.index):
    for col_i, mod in enumerate(pivot.columns):
        if pd.isna(pivot.loc[item, mod]):
            ax_heat.text(col_i + 0.5, row_i + 0.5, 'N/A',
                         ha='center', va='center', fontsize=8, color='#bbb')

ax_heat.set_title('Score par item × modèle')
ax_heat.set_xlabel('')
ax_heat.set_ylabel('')
ax_heat.tick_params(axis='x', rotation=25)

# Barplot difficulté
colors_item = ['#55A868' if v >= 0.8 else '#DD8452' if v >= 0.5 else '#C44E52'
               for v in item_means.values]
ax_item.barh(range(len(item_means)), item_means.values, color=colors_item, alpha=0.85)
ax_item.set_yticks(range(len(item_means)))
ax_item.set_yticklabels(item_means.index)
ax_item.set_xlim(0, 1.05)
ax_item.axvline(0.85, color='#55A868', linestyle='--', lw=1, label='0.85 = correct+optim')
ax_item.set_xlabel('Score moyen')
ax_item.set_title('Difficulté des tâches')
ax_item.legend(fontsize=8)
for i, v in enumerate(item_means.values):
    ax_item.text(v + 0.01, i, f'{v:.2f}', va='center', fontsize=8)

plt.tight_layout()
plt.show()

## 5. Anatomie du score — décomposition par composant

**Que montrent ces graphiques ?**  
Le score final n'est pas un monolithe — c'est une combinaison pondérée de 4 composants positifs et d'une pénalité. Ces graphiques montrent d'où vient chaque point dans la note finale.

**Formule :**
```
score = 0.50 × correction        ← la moitié du score vient de la correction pure
      + 0.20 × pbt_pass_rate      ← robustesse sur tests générés aléatoirement
      + 0.15 × optimization       ← bon régime de complexité (ex. O(n log n) et non O(n²))
      + 0.15 × pd_score           ← paradigme différent de l'oracle de référence
      − 0.25 × hallucination      ← pénalité pour crash, violation d'imports, ou test caché raté
```

**Comment lire le violin plot :** La largeur à une valeur donnée = densité des scores à cet endroit. Un pic à 0.85 = la plupart des solutions sont correctes+optimisées mais sans PD.

In [ ]:
df_ok = df_ok.copy()
pbt_col = 'pbt_pass_rate' if 'pbt_pass_rate' in df_ok.columns else None

df_ok['contrib_correct'] = 0.50 * df_ok['correctness_score'].fillna(0)
df_ok['contrib_pbt']     = 0.20 * (df_ok[pbt_col].fillna(0) if pbt_col else 0)
df_ok['contrib_optim']   = 0.15 * df_ok['optimization_score'].fillna(0)
df_ok['contrib_pd']      = 0.15 * df_ok['pd_score'].fillna(0)
df_ok['contrib_halluc']  = -0.25 * df_ok['hallucination_flag'].astype(float)

comp_by_model = (
    df_ok.groupby('model_short')
    [['contrib_correct','contrib_pbt','contrib_optim','contrib_pd','contrib_halluc']]
    .mean()
    .reindex([short(m) for m in model_order if short(m) in
              df_ok['model_short'].values])
)

fig, (ax_stk, ax_vio) = plt.subplots(1, 2, figsize=(15, 4.5))

# Stacked bar des contributions
comp_pos  = comp_by_model[['contrib_correct','contrib_pbt','contrib_optim','contrib_pd']]
comp_cols = ['#55A868','#64B5F6','#DD8452','#8172B3']
comp_labs = ['Correction (×0.50)','PBT robustesse (×0.20)','Optimisation (×0.15)','PD Score (×0.15)']

bot = np.zeros(len(comp_by_model))
for col, clr, lab in zip(comp_pos.columns, comp_cols, comp_labs):
    vals = comp_pos[col].values
    ax_stk.barh(range(len(comp_by_model)), vals, left=bot, color=clr, alpha=0.85, label=lab)
    bot += vals
ax_stk.barh(range(len(comp_by_model)), comp_by_model['contrib_halluc'].values,
            color='#C44E52', alpha=0.7, label='Hallucination (−0.25)')

ax_stk.set_yticks(range(len(comp_by_model)))
ax_stk.set_yticklabels(comp_by_model.index)
ax_stk.set_xlabel('Contribution au score final')
ax_stk.set_title('Décomposition du score par composant')
ax_stk.legend(loc='lower right', fontsize=8)
ax_stk.axvline(0, color='#999', lw=0.8)
ax_stk.set_xlim(-0.3, 1.0)

# Violin plot distribution des scores
score_data = []
labels_vio = []
for m in model_order:
    vals = df_ok[df_ok['model']==m]['score'].dropna().values
    if len(vals) >= 2:
        score_data.append(vals)
        labels_vio.append(short(m))

if score_data:
    parts = ax_vio.violinplot(score_data, vert=False, showmedians=True,
                               positions=range(len(score_data)))
    for i, (pc, m) in enumerate(zip(parts['bodies'],
                                     [m for m in model_order if m in df_ok['model'].values
                                      and len(df_ok[df_ok['model']==m]['score'].dropna()) >= 2])):
        pc.set_facecolor(color(m))
        pc.set_alpha(0.65)
    ax_vio.set_yticks(range(len(labels_vio)))
    ax_vio.set_yticklabels(labels_vio)

ax_vio.set_xlim(-0.05, 1.1)
ax_vio.set_xlabel('Score')
ax_vio.set_title('Distribution des scores (toutes tâches)')
ax_vio.axvline(0.85, color='#aaa', linestyle='--', lw=1, label='0.85 = seuil correct+optim')
ax_vio.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 6. Productive Divergence — exploration des 3 axes

**Qu'est-ce que la Productive Divergence (PD) ?**  
Un modèle fait preuve de PD quand il produit une solution **correcte** en utilisant un **paradigme algorithmique différent de l'oracle**.

Exemple : l'oracle résout la recherche de motif avec KMP (Knuth-Morris-Pratt). Un modèle qui utilise Rolling Hash à la place produit une vraie PD.

**Les 3 axes de mesure :**

| Axe | Ce que ça mesure | Seuil |
|---|---|---|
| `paradigm_distance` | À quel point la structure algorithmique diffère de l'oracle | < 0.20 = variation cosmétique uniquement |
| `productivity_score` | Efficacité temporelle × robustesse | — |
| `originality_score` | 1 − similarité tokenielle avec l'oracle | < 0.30 = quasi-copie |

`pd_score` = moyenne harmonique des 3 axes (seulement si les seuils sont respectés).

**Comment lire les scatter plots :**  
Chaque point = une solution correcte. **La taille du point = pd_score.** Les points dans le **coin supérieur droit** = vraie Productive Divergence.

In [ ]:
correct_df = df_ok[df_ok['correctness_score'] == 1.0].copy()
if 'originality_score' not in correct_df.columns and 'salieri_overlap' in correct_df.columns:
    correct_df['originality_score'] = 1.0 - correct_df['salieri_overlap']

fig = plt.figure(figsize=(16, 10))
gs  = GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# Scatter 1 : paradigm_distance vs originality
ax1 = fig.add_subplot(gs[0, 0])
for m in model_order:
    grp = correct_df[correct_df['model']==m]
    if grp.empty: continue
    sizes = (grp['pd_score'].fillna(0) * 300 + 30).clip(30, 400)
    ax1.scatter(grp.get('paradigm_distance', pd.Series([0]*len(grp))).fillna(0),
                grp.get('originality_score', pd.Series([0]*len(grp))).fillna(0),
                s=sizes, color=color(m), alpha=0.7, label=short(m), edgecolors='white', lw=0.8)
ax1.axvline(0.20, color='#aaa', ls=':', lw=1)
ax1.axhline(0.30, color='#aaa', ls=':', lw=1)
ax1.text(0.22, 0.02, 'cosmétique', fontsize=7, color='#aaa', rotation=90, va='bottom')
ax1.text(0.01, 0.32, 'copie oracle', fontsize=7, color='#aaa')
ax1.set_xlabel('paradigm_distance'); ax1.set_ylabel('originality_score')
ax1.set_title('Distance paradigme vs Originalité')
ax1.set_xlim(-0.05, 1.05); ax1.set_ylim(-0.05, 1.05)
ax1.legend(fontsize=7, loc='upper left')

# Scatter 2 : paradigm_distance vs productivity
ax2 = fig.add_subplot(gs[0, 1])
for m in model_order:
    grp = correct_df[correct_df['model']==m]
    if grp.empty: continue
    sizes = (grp['pd_score'].fillna(0) * 300 + 30).clip(30, 400)
    ax2.scatter(grp.get('paradigm_distance', pd.Series([0]*len(grp))).fillna(0),
                grp.get('productivity_score', pd.Series([0]*len(grp))).fillna(0),
                s=sizes, color=color(m), alpha=0.7, edgecolors='white', lw=0.8)
ax2.axvline(0.20, color='#aaa', ls=':', lw=1)
ax2.set_xlabel('paradigm_distance'); ax2.set_ylabel('productivity_score')
ax2.set_title('Distance paradigme vs Productivité')
ax2.set_xlim(-0.05, 1.05); ax2.set_ylim(-0.05, 1.05)

# Scatter 3 : originality vs productivity
ax3 = fig.add_subplot(gs[0, 2])
for m in model_order:
    grp = correct_df[correct_df['model']==m]
    if grp.empty: continue
    sizes = (grp['pd_score'].fillna(0) * 300 + 30).clip(30, 400)
    ax3.scatter(grp.get('originality_score', pd.Series([0]*len(grp))).fillna(0),
                grp.get('productivity_score', pd.Series([0]*len(grp))).fillna(0),
                s=sizes, color=color(m), alpha=0.7, edgecolors='white', lw=0.8)
ax3.axvline(0.30, color='#aaa', ls=':', lw=1)
ax3.set_xlabel('originality_score'); ax3.set_ylabel('productivity_score')
ax3.set_title('Originalité vs Productivité')
ax3.set_xlim(-0.05, 1.05); ax3.set_ylim(-0.05, 1.05)

# PD Score heatmap
ax4 = fig.add_subplot(gs[1, :2])
pivot_pd = df_ok.pivot_table(index='item_id', columns='model', values='pd_score', aggfunc='mean')
pivot_pd = pivot_pd.reindex(index=item_order, columns=model_order)
pivot_pd.columns = [short(m) for m in pivot_pd.columns]
sns.heatmap(pivot_pd.fillna(0), ax=ax4, cmap='YlOrRd', vmin=0, vmax=0.5,
            linewidths=0.5, linecolor='white',
            annot=pivot_pd.applymap(lambda v: f'{v:.2f}' if pd.notna(v) and v > 0.001 else '·'),
            fmt='', annot_kws={'size': 10},
            cbar_kws={'label': 'pd_score'})
ax4.set_title('pd_score par item × modèle  (· = pas de PD détectée)')
ax4.set_xlabel(''); ax4.set_ylabel('')
ax4.tick_params(axis='x', rotation=20)

# Histogramme pd_score
ax5 = fig.add_subplot(gs[1, 2])
for m in model_order:
    vals = correct_df[correct_df['model']==m]['pd_score'].dropna()
    if len(vals) == 0: continue
    ax5.hist(vals, bins=12, alpha=0.55, range=(0, 0.5), color=color(m), label=short(m))
ax5.set_xlabel('pd_score'); ax5.set_ylabel('Nb solutions')
ax5.set_title('Distribution du pd_score\n(solutions correctes seulement)')
ax5.legend(fontsize=7)

plt.suptitle('Productive Divergence — Analyse 3 axes  |  Taille des points ∝ pd_score',
             fontsize=12, y=1.01)
plt.show()

# Détail cas pd_score > 0
pd_nonzero = df_ok[df_ok['pd_score'] > 0][[
    'model','item_id','pd_score','paradigm_distance','productivity_score',
    'originality_score','candidate_paradigms','oracle_paradigms','genuine_divergence'
]].sort_values('pd_score', ascending=False)

print(f'\n── Solutions avec pd_score > 0 : {len(pd_nonzero)} ──')
if not pd_nonzero.empty:
    pd_nonzero = pd_nonzero.copy()
    pd_nonzero['model'] = pd_nonzero['model'].map(short)
    display(pd_nonzero.reset_index(drop=True).round(3))
else:
    print('Aucune PD détectée dans ce run.\n'
          'Interprétation : tous les modèles utilisent le même paradigme que l\'oracle.\n'
          'Ce n\'est pas nécessairement un problème — cela peut signifier que les paradigmes\n'
          'connus pour chaque item ne sont pas encore détectables par le classifier.')

## 7. True Regime — vérification empirique de la complexité

**Que mesure-t-on ici ?**  
Pour chaque solution, on mesure le ratio de temps d'exécution `t(grande entrée) / t(petite entrée)`.  
Ce ratio indique empiriquement la classe de complexité de la solution.

**Interprétation du ratio :**

| Ratio mesuré | Classe de complexité probable |
|---|---|
| ~ 1–2 | O(n) ou O(n log n) très rapide — très efficace |
| 2–30 | O(n log n) — attendu pour la plupart des items |
| 30–100 | O(n² / n log n) — **suspect** |
| > 100 | O(n²) ou pire — **hors contrainte** |

`complexity_ratio_ok = True` si le ratio est dans les bornes attendues pour l'item considéré.

**Note importante :** Un ratio < 1 est statistiquement possible (bruit Python, cache OS). Ce n'est pas physiquement réel mais reflète la variance de mesure sur de petites entrées.

In [ ]:
has_ratio = 'complexity_ratio' in df_ok.columns and df_ok['complexity_ratio'].notna().any()

if not has_ratio:
    print('Pas de données complexity_ratio dans ce run.')
else:
    ratio_df = df_ok.dropna(subset=['complexity_ratio']).copy()

    fig, (ax_box, ax_ok) = plt.subplots(1, 2, figsize=(15, 4.5))

    # Box plot par modèle
    models_r = [m for m in model_order if m in ratio_df['model'].values]
    data_box = [ratio_df[ratio_df['model']==m]['complexity_ratio'].values for m in models_r]
    bp = ax_box.boxplot(data_box, vert=False, patch_artist=True,
                         medianprops={'color':'black','lw':2},
                         flierprops={'marker':'o','markersize':4,'alpha':0.5})
    for patch, m in zip(bp['boxes'], models_r):
        patch.set_facecolor(color(m)); patch.set_alpha(0.7)

    ax_box.set_yticks(range(1, len(models_r)+1))
    ax_box.set_yticklabels([short(m) for m in models_r])
    ax_box.axvline(30, color='#C44E52', ls='--', lw=1.5, label='Limite O(n log n) ≈ 30×')
    ax_box.axvline(1,  color='#aaa', ls=':',  lw=1)
    ax_box.set_xlabel('Ratio temps large / petit')
    ax_box.set_title('Distribution du complexity_ratio')
    ax_box.set_xlim(0, max(40, ratio_df['complexity_ratio'].max() * 1.1))
    ax_box.legend(fontsize=8)

    # Heatmap complexity_ratio_ok
    pivot_cr = df_ok.pivot_table(index='item_id', columns='model',
                                  values='complexity_ratio_ok', aggfunc='mean')
    pivot_cr = pivot_cr.reindex(index=item_order, columns=model_order)
    pivot_cr.columns = [short(m) for m in pivot_cr.columns]
    sns.heatmap(pivot_cr.fillna(-0.1), ax=ax_ok,
                cmap='RdYlGn', vmin=0, vmax=1,
                linewidths=0.5, linecolor='white',
                annot=pivot_cr.applymap(lambda v: '✓' if v == 1 else '✗' if v == 0 else '—'),
                fmt='', annot_kws={'size': 12}, cbar=False)
    ax_ok.set_title('complexity_ratio_ok  (✓ = dans le bon régime)')
    ax_ok.set_xlabel(''); ax_ok.set_ylabel('')
    ax_ok.tick_params(axis='x', rotation=20)

    plt.tight_layout()
    plt.show()

    n_ok  = int(ratio_df['complexity_ratio_ok'].sum())
    n_tot = len(ratio_df)
    print(f'complexity_ratio_ok = True : {n_ok}/{n_tot} ({n_ok/n_tot*100:.0f}%)')
    print(f'Min={ratio_df["complexity_ratio"].min():.2f}  '
          f'Max={ratio_df["complexity_ratio"].max():.2f}  '
          f'Médiane={ratio_df["complexity_ratio"].median():.2f}')

    bad = ratio_df[ratio_df['complexity_ratio'] > 30]
    if not bad.empty:
        print('\n⚠ Solutions hors régime (ratio > 30) :')
        display(bad[['model','item_id','complexity_ratio','correctness_score']]
                .assign(model=lambda d: d['model'].map(short)).reset_index(drop=True))
    else:
        print('\n✓ Aucune solution hors régime (ratio > 30) dans ce run.')
        print('⚠ Attention : si tous les ratios sont proches de 1, les tailles de stress test'
              ' sont peut-être trop petites pour discriminer O(n²) de O(n log n).')

## 8. Analyse des échecs

**Que montre cette section ?**  
Tous les cas d'échec, catégorisés par type. Il faut distinguer :

| Type d'échec | Cause | Ce que ça indique |
|---|---|---|
| **Erreur API** | Rate limit (429), timeout réseau | Problème infrastructure, ne reflète pas la qualité du modèle |
| **Violation statique** | Import interdit ou syntax error | Le modèle a généré du code invalide |
| **Crash Python** | Exception à l'exécution | Bug logique dans le code généré |
| **Timeout** | Trop lent sur les grands inputs | Solution O(n²) là où O(n log n) est requis |
| **Hallucination (hidden)** | Tests publics OK mais tests cachés échoués | La solution n'est correcte que sur les exemples visibles |

**Le graphique de droite** montre le gap public/hidden : les points sous la diagonale = solutions qui surfit les tests publics mais échouent sur les tests cachés.

In [ ]:
def failure_type(row):
    if row.get('is_error', False):    return 'Erreur API'
    if row.get('static_violation'):   return 'Violation statique'
    if row.get('crash'):              return 'Crash Python'
    if row.get('timeout'):            return 'Timeout'
    if row.get('hallucination_flag'): return 'Hallucination (hidden)'
    if row.get('correctness_score', 1) < 1: return 'Tests échoués'
    return None

df['failure_type'] = df.apply(failure_type, axis=1)
failures = df[df['failure_type'].notna()].copy()

fail_counts = failures.groupby(['model_short','failure_type']).size().reset_index(name='count')
fail_pivot  = fail_counts.pivot(index='model_short', columns='failure_type', values='count').fillna(0)

fig, (ax_bar_f, ax_gap) = plt.subplots(1, 2, figsize=(15, 4.5))

# Stacked bar des échecs
fail_colors = {
    'Erreur API':'#B0BEC5', 'Violation statique':'#FF8A65',
    'Crash Python':'#EF5350', 'Timeout':'#FFA726',
    'Hallucination (hidden)':'#AB47BC', 'Tests échoués':'#EC407A'
}
if not fail_pivot.empty:
    bot = np.zeros(len(fail_pivot))
    for col in fail_pivot.columns:
        vals = fail_pivot[col].values
        ax_bar_f.barh(range(len(fail_pivot)), vals, left=bot,
                      color=fail_colors.get(col, '#888'), alpha=0.85, label=col)
        bot += vals
    ax_bar_f.set_yticks(range(len(fail_pivot)))
    ax_bar_f.set_yticklabels(fail_pivot.index)
    ax_bar_f.set_xlabel("Nombre d'échecs")
    ax_bar_f.set_title('Échecs par type et par modèle')
    ax_bar_f.legend(loc='lower right', fontsize=8, bbox_to_anchor=(1.0, -0.05))
else:
    ax_bar_f.text(0.5, 0.5, 'Aucun échec', ha='center', va='center',
                  transform=ax_bar_f.transAxes, fontsize=12)

# Public vs Hidden gap
gap_df = df_ok[df_ok['public_pass_rate'].notna() & df_ok['hidden_pass_rate'].notna()].copy()
if not gap_df.empty:
    for m in model_order:
        grp = gap_df[gap_df['model']==m]
        if grp.empty: continue
        ax_gap.scatter(grp['public_pass_rate'], grp['hidden_pass_rate'],
                       s=60, color=color(m), alpha=0.75, label=short(m),
                       edgecolors='white', lw=0.8)
    ax_gap.plot([0,1],[0,1],'k--',lw=1,alpha=0.4,label='Public = Hidden')
    ax_gap.fill_between([0,1],[0,1],[0,0],alpha=0.04,color='#C44E52')
    ax_gap.text(0.6, 0.08, 'Over-fitting\n(public > hidden)',
                fontsize=8, color='#C44E52', alpha=0.8)
    ax_gap.set_xlabel('Public pass rate')
    ax_gap.set_ylabel('Hidden pass rate')
    ax_gap.set_title('Gap public / tests cachés\n(sous la diagonale = sur-apprentissage)')
    ax_gap.set_xlim(-0.05,1.05); ax_gap.set_ylim(-0.05,1.05)
    ax_gap.legend(fontsize=8, loc='upper left')
else:
    ax_gap.text(0.5, 0.5, 'pass_rate indisponibles', ha='center', va='center',
                transform=ax_gap.transAxes)

plt.tight_layout()
plt.show()

critical = failures[failures['failure_type'] != 'Erreur API'][[
    'model_short','item_id','failure_type','score','correctness_score',
    'hidden_pass_rate','static_violations'
]].rename(columns={
    'model_short':'Modèle','item_id':'Item','failure_type':'Type','score':'Score',
    'correctness_score':'Correct','hidden_pass_rate':'Hidden %','static_violations':'Violation'
})
if not critical.empty:
    print(f'\n── Échecs critiques (hors erreurs API) : {len(critical)} ──')
    display(critical.sort_values('Type').reset_index(drop=True))

## 9. Récapitulatif final

Tableau de synthèse et points clés automatiques du run.

In [ ]:
summary = (
    df_ok.groupby('model')
    .agg(
        N             =('item_id', 'count'),
        Score         =('score', 'mean'),
        Correction    =('correctness_score', 'mean'),
        Optimisation  =('optimization_score', 'mean'),
        PD_Score      =('pd_score', 'mean'),
        Hallucination =('hallucination_flag', 'mean'),
        Genuine_PD    =('genuine_divergence', 'mean'),
    )
    .sort_values('Score', ascending=False)
    .reset_index()
)
summary.insert(0, 'Modèle', summary['model'].map(short))
display(summary[['Modèle','N','Score','Correction','Optimisation','PD_Score','Hallucination','Genuine_PD']].round(3))

print('\n── Points clés du run ──')
best  = summary.iloc[0]
worst = summary.iloc[-1]
print(f'  🏆 Meilleur modèle  : {best["Modèle"]} (score={best["Score"]:.2f}, correction={best["Correction"]:.0%})')
print(f'  ⚠  Moins bon modèle : {worst["Modèle"]} (score={worst["Score"]:.2f})')
n_pd_total = int((df_ok['pd_score']>0).sum())
n_genuine  = int(df_ok['genuine_divergence'].sum())
print(f'  🔀 Productive Divergence : {n_pd_total} solutions PD (dont {n_genuine} genuine_divergence=True)')
if has_ratio:
    n_ok_r = int(df_ok['complexity_ratio_ok'].sum())
    n_r    = int(df_ok['complexity_ratio'].notna().sum())
    print(f'  ⏱  True Regime OK : {n_ok_r}/{n_r} solutions ({n_ok_r/n_r*100:.0f}%)')
n_halluc = int(df_ok['hallucination_flag'].sum())
print(f'  👻 Hallucinations  : {n_halluc}/{len(df_ok)} ({n_halluc/len(df_ok)*100:.0f}%)')
n_err = int(df['is_error'].sum())
if n_err > 0:
    print(f'  🔌 Erreurs API     : {n_err} (ne comptent pas dans les scores)')